## PC1 - Programación Concurrente y Distribuida (1ACC0065)
**Generación de Dataset Sintético:** Sistema de Recomendación de Microcursos Adaptativos (Matemática, Lectura y Ciencias)
Este script genera un dataset sintético de interacciones estudiante-curso (>1,000,000 registros) sobre una estructura curricular representada como un Grafo Acíclico Dirigido (DAG) de competencias, tal como lo exige el objetivo específico 2 y 3 del proyecto.


### IMPORTS Y CONFIGURACIÓN GLOBAL

In [ ]:
# En esta celda se importan las librerías necesarias y se fijan los
# parámetros globales de la generación (semilla aleatoria, tamaño del
# dataset, número de estudiantes, número de microcursos por área, etc.)

import numpy as np
import pandas as pd
import random
import string
import datetime as dt
from dataclasses import dataclass, field

# Semilla para reproducibilidad (importante para poder comparar
# versión secuencial vs concurrente en la PC2 sobre el MISMO dataset)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Parámetros del dataset ---------------------------------------------
N_STUDENTS = 25_000          # número de estudiantes simulados
N_INTERACTIONS_TARGET = 1_200_000   # > 1,000,000 registros (requisito del PC1)

# Núcleos fundamentales mencionados en el resumen del proyecto
NUCLEOS = ["Matematica", "Lectura", "Ciencias"]

# Número de microcursos ("unidades mínimas de conocimiento") por núcleo.
# Se generan varios niveles de profundidad para poder construir un DAG
# de prerrequisitos realista (currículo atomizado).
N_MICROCURSOS_POR_NUCLEO = 60
N_NIVELES = 6   # niveles de profundidad del DAG (0 = fundamentos, 5 = avanzado)

print("Configuración cargada.")
print(f"Estudiantes: {N_STUDENTS:,}")
print(f"Interacciones objetivo: {N_INTERACTIONS_TARGET:,}")
print(f"Núcleos: {NUCLEOS}")


Configuración cargada.
Estudiantes: 25,000
Interacciones objetivo: 1,200,000
Núcleos: ['Matematica', 'Lectura', 'Ciencias']


### GENERACIÓN DEL CATÁLOGO DE MICROCURSOS Y DEL DAG DE COMPETENCIAS

In [ ]:
# Cada microcurso pertenece a un núcleo (Matemática, Lectura, Ciencias) y a
# un nivel (0 a N_NIVELES-1). Un microcurso de nivel k puede tener como
# prerrequisitos 1 o 2 microcursos de niveles estrictamente menores del
# MISMO núcleo. Esto garantiza que el grafo resultante sea acíclico (DAG),
# ya que las aristas siempre apuntan de un nivel menor a uno mayor.
#
# Esta estructura es la que luego, en la implementación real (PC2), se
# recorrerá de forma concurrente (lecturas masivas) para validar qué
# microcursos ya están "desbloqueados" para cada estudiante.

TEMAS_POR_NUCLEO = {
    "Matematica": [
        "Numeros naturales", "Fracciones", "Decimales", "Proporcionalidad",
        "Ecuaciones lineales", "Ecuaciones cuadraticas", "Funciones",
        "Geometria plana", "Geometria del espacio", "Trigonometria",
        "Estadistica descriptiva", "Probabilidad", "Combinatoria",
        "Limites", "Derivadas", "Integrales", "Matrices", "Vectores",
        "Logica matematica", "Sucesiones y series",
    ],
    "Lectura": [
        "Comprension literal", "Comprension inferencial", "Comprension critica",
        "Vocabulario en contexto", "Estructura del texto narrativo",
        "Estructura del texto argumentativo", "Identificacion de ideas principales",
        "Analisis de textos expositivos", "Interpretacion de graficos y tablas",
        "Lectura de poesia", "Analisis de textos periodisticos",
        "Deteccion de sesgos y falacias", "Sintesis de multiples fuentes",
        "Redaccion de resumenes", "Analisis de textos historicos",
        "Comprension de instrucciones tecnicas", "Lectura de textos cientificos",
        "Retorica y persuasion", "Analisis comparativo de autores",
        "Lectura critica de medios digitales",
    ],
    "Ciencias": [
        "Metodo cientifico", "Materia y energia", "Celula y organizacion biologica",
        "Genetica basica", "Ecosistemas", "Fisica del movimiento",
        "Fuerzas y leyes de Newton", "Electricidad basica", "Quimica de la materia",
        "Reacciones quimicas", "Sistema solar y astronomia", "Geologia basica",
        "Cambio climatico", "Cuerpo humano", "Evolucion biologica",
        "Ondas y sonido", "Optica basica", "Termodinamica basica",
        "Quimica organica introductoria", "Pensamiento cientifico y evidencia",
    ],
}

def generar_catalogo_microcursos():
    """Genera el catálogo de microcursos con nivel y núcleo asignado."""
    microcursos = []
    course_id = 0
    for nucleo in NUCLEOS:
        temas = TEMAS_POR_NUCLEO[nucleo]
        for i in range(N_MICROCURSOS_POR_NUCLEO):
            nivel = i % N_NIVELES  # distribuye microcursos entre los niveles
            tema_base = temas[i % len(temas)]
            microcursos.append({
                "course_id": f"{nucleo[:3].upper()}_{course_id:04d}",
                "nucleo": nucleo,
                "nivel": nivel,
                "titulo": f"{tema_base} - Modulo {i // len(temas) + 1}",
                "duracion_min": int(np.random.choice([10, 15, 20, 25, 30])),
            })
            course_id += 1
    return pd.DataFrame(microcursos)


def generar_dag_prerequisitos(df_cursos: pd.DataFrame):
    """
    Construye las aristas del DAG de competencias: (prerrequisito -> curso).
    Solo se conectan microcursos del mismo núcleo, y siempre desde un nivel
    menor hacia un nivel mayor (esto asegura que no haya ciclos).
    """
    aristas = []
    for nucleo in NUCLEOS:
        sub = df_cursos[df_cursos["nucleo"] == nucleo]
        por_nivel = {n: sub[sub["nivel"] == n]["course_id"].tolist()
                     for n in range(N_NIVELES)}
        for nivel in range(1, N_NIVELES):
            cursos_nivel = por_nivel[nivel]
            candidatos_prev = por_nivel[nivel - 1]
            if not candidatos_prev:
                continue
            for curso in cursos_nivel:
                # cada curso tiene entre 1 y 2 prerrequisitos del nivel anterior
                n_prereqs = random.choice([1, 1, 2])
                prereqs = random.sample(
                    candidatos_prev, k=min(n_prereqs, len(candidatos_prev))
                )
                for p in prereqs:
                    aristas.append({
                        "prerequisito_id": p,
                        "course_id": curso,
                        "nucleo": nucleo,
                    })
    return pd.DataFrame(aristas)


df_cursos = generar_catalogo_microcursos()
df_dag = generar_dag_prerequisitos(df_cursos)

print(f"Microcursos generados: {len(df_cursos):,}")
print(f"Aristas del DAG (prerrequisitos): {len(df_dag):,}")
df_cursos.head()


Microcursos generados: 180
Aristas del DAG (prerrequisitos): 197


,course_id,nucleo,nivel,titulo,duracion_min
0,MAT_0000,Matematica,0,Numeros naturales - Modulo 1,25
1,MAT_0001,Matematica,1,Fracciones - Modulo 1,30
2,MAT_0002,Matematica,2,Decimales - Modulo 1,20
3,MAT_0003,Matematica,3,Proporcionalidad - Modulo 1,30
4,MAT_0004,Matematica,4,Ecuaciones lineales - Modulo 1,30


### VALIDACIÓN DE ACICLICIDAD DEL GRAFO DE COMPETENCIAS

In [ ]:
# Verificamos con un ordenamiento topológico simple (Kahn) que el grafo
# generado efectivamente es un DAG. Esto es importante porque el objetivo
# específico 3 exige explícitamente estructurar el conocimiento como un
# grafo ACÍCLICO dirigido.

def es_dag(df_cursos: pd.DataFrame, df_dag: pd.DataFrame) -> bool:
    from collections import defaultdict, deque

    grado_entrada = {cid: 0 for cid in df_cursos["course_id"]}
    adyacencia = defaultdict(list)

    for _, fila in df_dag.iterrows():
        adyacencia[fila["prerequisito_id"]].append(fila["course_id"])
        grado_entrada[fila["course_id"]] += 1

    cola = deque([cid for cid, g in grado_entrada.items() if g == 0])
    visitados = 0

    while cola:
        actual = cola.popleft()
        visitados += 1
        for vecino in adyacencia[actual]:
            grado_entrada[vecino] -= 1
            if grado_entrada[vecino] == 0:
                cola.append(vecino)

    return visitados == len(df_cursos)


assert es_dag(df_cursos, df_dag), "El grafo de competencias generado NO es un DAG"
print("Verificación superada: el grafo de competencias es un DAG válido (sin ciclos).")


Verificación superada: el grafo de competencias es un DAG válido (sin ciclos).


### GENERACIÓN DE PERFILES DE ESTUDIANTES

In [ ]:
# Cada estudiante recibe:
#   - una afinidad (interés) hacia cada núcleo, en [0,1], que modela sus
#     "intereses personales" (alineado con el enfoque de rutas de interés
#     descrito en el resumen del proyecto).
#   - una habilidad base por núcleo, en [0,1], que influye en su desempeño
#     (nota) al completar un microcurso.
# Estas dos variables generan la señal que luego un sistema de filtrado
# colaborativo (o similitud coseno) podría aprender a partir del dataset.

def generar_estudiantes(n_estudiantes: int) -> pd.DataFrame:
    ids = [f"STU_{i:06d}" for i in range(n_estudiantes)]

    # Afinidad por núcleo (interés): se generan con Dirichlet para que
    # cada estudiante tenga un perfil de interés distinto pero comparable.
    afinidades = np.random.dirichlet(alpha=[1.5, 1.5, 1.5], size=n_estudiantes)

    # Habilidad base por núcleo (rendimiento esperado), correlacionada
    # débilmente con la afinidad (a mayor interés, ligera mejora de habilidad)
    habilidad_base = np.random.beta(a=2, b=2, size=(n_estudiantes, 3))
    habilidad = np.clip(habilidad_base + 0.15 * afinidades, 0, 1)

    grados = np.random.choice(
        ["Primaria alta", "Secundaria baja", "Secundaria alta"],
        size=n_estudiantes, p=[0.25, 0.4, 0.35]
    )

    df = pd.DataFrame({
        "student_id": ids,
        "grado": grados,
        "afinidad_matematica": afinidades[:, 0],
        "afinidad_lectura": afinidades[:, 1],
        "afinidad_ciencias": afinidades[:, 2],
        "habilidad_matematica": habilidad[:, 0],
        "habilidad_lectura": habilidad[:, 1],
        "habilidad_ciencias": habilidad[:, 2],
    })
    return df


df_estudiantes = generar_estudiantes(N_STUDENTS)
print(f"Estudiantes generados: {len(df_estudiantes):,}")
df_estudiantes.head()


Estudiantes generados: 25,000


,student_id,grado,afinidad_matematica,afinidad_lectura,afinidad_ciencias,habilidad_matematica,habilidad_lectura,habilidad_ciencias
0,STU_000000,Secundaria alta,0.177757,0.255714,0.566529,0.117477,0.436960,0.745640
1,STU_000001,Primaria alta,0.244428,0.481775,0.273797,0.165711,0.908680,0.379621
2,STU_000002,Primaria alta,0.369477,0.106502,0.524021,0.707814,0.645356,0.770228
3,STU_000003,Secundaria alta,0.116829,0.698441,0.184730,0.735986,0.612264,0.704721
4,STU_000004,Secundaria alta,0.502720,0.097188,0.400091,0.610454,0.352486,0.274789


### GENERACIÓN DE INTERACCIONES ESTUDIANTE-MICROCURSO (>1,000,000 REGISTROS)

In [ ]:
# Este es el núcleo del dataset: cada fila representa la interacción de un
# estudiante con un microcurso (evento de tipo "intento"). Se simula de
# forma vectorizada (NumPy) para poder generar >1M filas eficientemente.
#
# Reglas de negocio simuladas (para que el dataset sea coherente con el
# caso de uso, no puramente aleatorio):
#   1. La probabilidad de que un estudiante elija un curso de un núcleo
#      depende de su afinidad hacia ese núcleo (rutas de interés).
#   2. Dentro de un núcleo, se prioriza tomar cursos de nivel más bajo
#      primero (progresión curricular sobre el DAG), con cierta
#      probabilidad de explorar niveles más altos (esto se usará luego
#      para simular estudiantes que "saltan" prerrequisitos, un caso de
#      interés para el sistema de recomendación).
#   3. La nota/desempeño depende de la habilidad del estudiante en ese
#      núcleo más ruido aleatorio.
#   4. El tiempo dedicado depende de la duración base del microcurso y
#      de si el estudiante aprobó o no (reintentos).

nucleo_a_afinidad_col = {
    "Matematica": "afinidad_matematica",
    "Lectura": "afinidad_lectura",
    "Ciencias": "afinidad_ciencias",
}
nucleo_a_habilidad_col = {
    "Matematica": "habilidad_matematica",
    "Lectura": "habilidad_lectura",
    "Ciencias": "habilidad_ciencias",
}

# Pre-indexar cursos por nucleo y nivel para un muestreo rápido
cursos_por_nucleo_nivel = {
    (row.nucleo, row.nivel): []
    for row in df_cursos.itertuples()
}
for row in df_cursos.itertuples():
    cursos_por_nucleo_nivel[(row.nucleo, row.nivel)].append(row.course_id)

fecha_inicio = dt.datetime(2026, 3, 1)  # inicio de año escolar


def generar_interacciones(n_filas: int) -> pd.DataFrame:
    # 1) Elegir estudiante para cada interacción (algunos estudiantes son
    #    más activos que otros -> distribución long-tail con Zipf-like)
    pesos_actividad = np.random.pareto(a=2.0, size=N_STUDENTS) + 1
    pesos_actividad /= pesos_actividad.sum()
    idx_estudiantes = np.random.choice(N_STUDENTS, size=n_filas, p=pesos_actividad)

    afinidades = df_estudiantes[
        ["afinidad_matematica", "afinidad_lectura", "afinidad_ciencias"]
    ].to_numpy()
    habilidades = df_estudiantes[
        ["habilidad_matematica", "habilidad_lectura", "habilidad_ciencias"]
    ].to_numpy()

    afin_estudiante = afinidades[idx_estudiantes]     # (n_filas, 3)
    habil_estudiante = habilidades[idx_estudiantes]   # (n_filas, 3)

    # 2) Elegir núcleo de la interacción en función de la afinidad de
    #    cada estudiante (muestreo categórico fila por fila, vectorizado
    #    mediante el truco de la CDF acumulada + un número aleatorio)
    cdf = afin_estudiante.cumsum(axis=1)
    r = np.random.rand(n_filas, 1)
    nucleo_idx = (r > cdf[:, :2]).sum(axis=1)  # 0=Mate, 1=Lectura, 2=Ciencias
    nucleos_elegidos = np.array(NUCLEOS)[nucleo_idx]

    # 3) Elegir nivel del microcurso: se favorece progresión (niveles bajos)
    #    con una geométrica truncada al rango [0, N_NIVELES-1]
    niveles = np.random.geometric(p=0.45, size=n_filas) - 1
    niveles = np.clip(niveles, 0, N_NIVELES - 1)

    # 4) Elegir el microcurso específico dentro de (nucleo, nivel)
    course_ids = np.empty(n_filas, dtype=object)
    for nucleo in NUCLEOS:
        for nivel in range(N_NIVELES):
            mask = (nucleos_elegidos == nucleo) & (niveles == nivel)
            n_match = mask.sum()
            if n_match == 0:
                continue
            opciones = cursos_por_nucleo_nivel[(nucleo, nivel)]
            course_ids[mask] = np.random.choice(opciones, size=n_match)

    # 5) Habilidad relevante del estudiante para el núcleo elegido
    habil_relevante = habil_estudiante[np.arange(n_filas), nucleo_idx]

    # 6) Nota/desempeño simulado: habilidad + ruido gaussiano, recortado [0,20]
    ruido = np.random.normal(loc=0, scale=3.0, size=n_filas)
    nota = np.clip(habil_relevante * 20 + ruido, 0, 20).round(1)

    # 7) Aprobado si nota >= 11 (escala vigesimal, sistema educativo peruano)
    aprobado = (nota >= 11).astype(int)

    # 8) Duración real: duración base del curso * factor según desempeño
    duracion_base = df_cursos.set_index("course_id")["duracion_min"]
    duracion_base_arr = duracion_base.loc[course_ids].to_numpy()
    factor_tiempo = np.where(aprobado == 1,
                              np.random.uniform(0.8, 1.2, n_filas),
                              np.random.uniform(1.0, 1.8, n_filas))
    tiempo_dedicado_min = np.round(duracion_base_arr * factor_tiempo, 1)

    # 9) Número de intento (algunos estudiantes reintentan el mismo curso)
    intento = np.random.choice([1, 2, 3], size=n_filas, p=[0.75, 0.18, 0.07])

    # 10) Timestamp de la interacción a lo largo de ~9 meses de año escolar
    offsets_dias = np.random.randint(0, 270, size=n_filas)
    offsets_horas = np.random.randint(0, 24, size=n_filas)
    timestamps = [
        fecha_inicio + dt.timedelta(days=int(d), hours=int(h))
        for d, h in zip(offsets_dias, offsets_horas)
    ]

    # 11) Dispositivo de acceso (variable adicional típica de plataformas e-learning)
    dispositivo = np.random.choice(
        ["movil", "laptop", "tablet", "desktop"],
        size=n_filas, p=[0.45, 0.30, 0.15, 0.10]
    )

    df_interacciones = pd.DataFrame({
        "interaction_id": np.arange(1, n_filas + 1),
        "student_id": df_estudiantes["student_id"].to_numpy()[idx_estudiantes],
        "course_id": course_ids,
        "nucleo": nucleos_elegidos,
        "nivel_curso": niveles,
        "intento": intento,
        "nota": nota,
        "aprobado": aprobado,
        "tiempo_dedicado_min": tiempo_dedicado_min,
        "dispositivo": dispositivo,
        "timestamp": timestamps,
    })
    return df_interacciones


df_interacciones = generar_interacciones(N_INTERACTIONS_TARGET)
print(f"Interacciones generadas: {len(df_interacciones):,}")
df_interacciones.head()


Interacciones generadas: 1,200,000


,interaction_id,student_id,course_id,nucleo,nivel_curso,intento,nota,aprobado,tiempo_dedicado_min,dispositivo,timestamp
0,1,STU_015170,CIE_0138,Ciencias,0,1,11.4,1,14.8,laptop,2026-05-23 02:00:00
1,2,STU_003594,CIE_0139,Ciencias,1,2,20.0,1,35.0,movil,2026-11-18 02:00:00
2,3,STU_024950,MAT_0043,Matematica,1,1,1.6,0,19.2,movil,2026-11-05 08:00:00
3,4,STU_005245,MAT_0012,Matematica,0,1,10.2,0,34.9,movil,2026-08-31 12:00:00
4,5,STU_018789,LEC_0114,Lectura,0,2,14.6,1,10.3,movil,2026-10-29 01:00:00


### LIMPIEZA Y DEPURACIÓN DEL DATASET

In [ ]:
# El objetivo específico 2 pide explícitamente "seleccionar y DEPURAR" el
# dataset. Aquí se aplican validaciones típicas de un pipeline real:
#   - eliminar duplicados exactos
#   - verificar que todas las llaves foráneas (student_id, course_id)
#     existan en sus tablas maestras
#   - verificar rangos válidos de las variables numéricas
#   - ordenar cronológicamente

def depurar_dataset(df_inter: pd.DataFrame,
                     df_est: pd.DataFrame,
                     df_curso: pd.DataFrame) -> pd.DataFrame:
    n_inicial = len(df_inter)

    # 1) eliminar duplicados exactos
    df_inter = df_inter.drop_duplicates()

    # 2) integridad referencial
    estudiantes_validos = set(df_est["student_id"])
    cursos_validos = set(df_curso["course_id"])
    df_inter = df_inter[
        df_inter["student_id"].isin(estudiantes_validos)
        & df_inter["course_id"].isin(cursos_validos)
    ]

    # 3) rangos válidos
    df_inter = df_inter[
        (df_inter["nota"] >= 0) & (df_inter["nota"] <= 20)
        & (df_inter["tiempo_dedicado_min"] > 0)
        & (df_inter["intento"].between(1, 3))
    ]

    # 4) tipos de datos consistentes
    df_inter["aprobado"] = df_inter["aprobado"].astype(int)
    df_inter["nivel_curso"] = df_inter["nivel_curso"].astype(int)

    # 5) orden cronológico y reindexado del id de interacción
    df_inter = df_inter.sort_values("timestamp").reset_index(drop=True)
    df_inter["interaction_id"] = np.arange(1, len(df_inter) + 1)

    n_final = len(df_inter)
    print(f"Registros antes de depurar: {n_inicial:,}")
    print(f"Registros después de depurar: {n_final:,}")
    print(f"Registros eliminados: {n_inicial - n_final:,}")

    return df_inter


df_interacciones = depurar_dataset(df_interacciones, df_estudiantes, df_cursos)

# Verificación explícita del requisito de >1,000,000 registros
assert len(df_interacciones) > 1_000_000, "El dataset depurado no supera 1,000,000 de registros"
print("Requisito cumplido: el dataset depurado supera 1,000,000 de registros.")


Registros antes de depurar: 1,200,000
Registros después de depurar: 1,200,000
Registros eliminados: 0
Requisito cumplido: el dataset depurado supera 1,000,000 de registros.


### ESTADÍSTICAS DESCRIPTIVAS Y VALIDACIÓN DE PERTINENCIA

In [ ]:
# Se generan estadísticas simples que sustentan que el dataset representa
# adecuadamente el caso de uso: distribución balanceada entre los 3
# núcleos, relación esperada entre afinidad y participación, etc.

print("=== Distribución de interacciones por núcleo ===")
print(df_interacciones["nucleo"].value_counts(normalize=True).round(3))

print("\n=== Tasa de aprobación por núcleo ===")
print(df_interacciones.groupby("nucleo")["aprobado"].mean().round(3))

print("\n=== Nota promedio por núcleo ===")
print(df_interacciones.groupby("nucleo")["nota"].mean().round(2))

print("\n=== Interacciones por nivel de curso (progresión sobre el DAG) ===")
print(df_interacciones["nivel_curso"].value_counts().sort_index())

print("\n=== Actividad de estudiantes (long-tail esperado) ===")
actividad = df_interacciones["student_id"].value_counts()
print(actividad.describe().round(2))


=== Distribución de interacciones por núcleo ===
nucleo
Ciencias      0.334
Matematica    0.333
Lectura       0.333
Name: proportion, dtype: float64

=== Tasa de aprobación por núcleo ===
nucleo
Ciencias      0.530
Lectura       0.520
Matematica    0.526
Name: aprobado, dtype: float64

=== Nota promedio por núcleo ===
nucleo
Ciencias      11.30
Lectura       11.19
Matematica    11.26
Name: nota, dtype: float64

=== Interacciones por nivel de curso (progresión sobre el DAG) ===
nivel_curso
0    539875
1    297324
2    163121
3     89996
4     49457
5     60227
Name: count, dtype: int64

=== Actividad de estudiantes (long-tail esperado) ===
count    25000.00
mean        48.00
std         55.47
min          7.00
25%         28.00
50%         35.00
75%         49.00
max       2051.00
Name: count, dtype: float64


### EXPORTACIÓN DE LOS ARCHIVOS FINALES DEL DATASET

In [ ]:
# Se exportan 3 archivos, que en conjunto conforman el dataset sintético:
#   1. cursos.csv          -> catálogo de microcursos (nodos del DAG)
#   2. dag_prerequisitos.csv -> aristas del DAG de competencias
#   3. estudiantes.csv     -> perfiles de estudiantes
#   4. interacciones.csv   -> tabla principal de hechos (>1M filas)
#
# En Colab, tras correr esta celda, los archivos quedarán en el entorno
# de ejecución y se pueden descargar con:
#   from google.colab import files
#   files.download("interacciones.csv")

OUTPUT_DIR = "."  # en Colab, dejar "." para guardar en /content

df_cursos.to_csv(f"{OUTPUT_DIR}/cursos.csv", index=False)
df_dag.to_csv(f"{OUTPUT_DIR}/dag_prerequisitos.csv", index=False)
df_estudiantes.to_csv(f"{OUTPUT_DIR}/estudiantes.csv", index=False)

# Para el archivo grande de interacciones se usa Parquet además de CSV,
# ya que Parquet es mucho más liviano y rápido de leer para >1M filas
# (recomendado para las siguientes etapas del proyecto, PC2/PC3).
df_interacciones.to_csv(f"{OUTPUT_DIR}/interacciones.csv", index=False)
df_interacciones.to_parquet(f"{OUTPUT_DIR}/interacciones.parquet", index=False)

print("Archivos exportados:")
print(" - cursos.csv")
print(" - dag_prerequisitos.csv")
print(" - estudiantes.csv")
print(" - interacciones.csv")
print(" - interacciones.parquet")
print(f"\nTotal de registros en interacciones: {len(df_interacciones):,}")


Archivos exportados:
 - cursos.csv
 - dag_prerequisitos.csv
 - estudiantes.csv
 - interacciones.csv
 - interacciones.parquet

Total de registros en interacciones: 1,200,000


### (OPCIONAL) DESCARGA DIRECTA DESDE GOOGLE COLAB

In [ ]:
# Descomentar este bloque si se ejecuta dentro de Google Colab y se desea
# descargar los archivos directamente al computador local.

from google.colab import files
files.download(f"{OUTPUT_DIR}/cursos.csv")
files.download(f"{OUTPUT_DIR}/dag_prerequisitos.csv")
files.download(f"{OUTPUT_DIR}/estudiantes.csv")
files.download(f"{OUTPUT_DIR}/interacciones.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>